# 2 · Optimization: reranker distillation + AWQ 4-bit, RAGAS before vs after

**Kaggle · GPU T4 x2 · Internet ON.**
1. **Distil** `bge-reranker-v2-m3` (568M) into `multilingual-e5-small` (118M) — done first, while the GPUs are free.
2. **AWQ**: serve `Qwen/Qwen3-8B-AWQ`, answer all questions (`awq` = AWQ + teacher reranker, `awq_student` = AWQ + distilled reranker).
3. **fp16 baseline**: serve `Qwen/Qwen3-8B`, answer the same questions, then judge *all* answer sets with the fp16 model.
4. Report: RAGAS before/after, latency p50/p95, TTFT, throughput, weight memory -> `reports/quantization.md`, MLflow.
Budget ~2-3 GPU-hours.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/AhmedYasser06/Egyptian-Civil-Code-RAG.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1

## Corpus
`dvc pull` fetches the PDF, the JSON corpus, the chunks and the embeddings from the shared DVC remote.
If the remote needs credentials, add Kaggle secrets `DVC_USER` / `DVC_PASSWORD` (Add-ons -> Secrets).
Fallback: attach a Kaggle dataset containing your `data/processed/` folder.

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
try:
    s = UserSecretsClient()
    user, pw = s.get_secret("DVC_USER"), s.get_secret("DVC_PASSWORD")
    !pip install -q "dvc==3.67.1" "dvc-s3==3.3.0"
    !dvc remote modify --local origin auth basic
    !dvc remote modify --local origin user {user}
    !dvc remote modify --local origin password {pw}
except Exception as e:
    print("no DVC secrets, trying anonymous pull:", e)
    !pip install -q "dvc==3.67.1" "dvc-s3==3.3.0"
!dvc pull
!ls -la data/processed
# fallback if dvc pull failed:  !mkdir -p data/processed && cp /kaggle/input/*/final-*.json /kaggle/input/*/embedded-*.json data/processed/

## vLLM in its own environment
vLLM pins its own torch/transformers, so it gets a separate venv (made with `uv`: Kaggle's `python -m venv` has no ensurepip). Kaggle doesn't allow `!cmd &`, so servers start with `subprocess.Popen`.

In [ ]:
!pip install -q uv
!uv venv /kaggle/vllm-env --python 3.11
!uv pip install -q --python /kaggle/vllm-env/bin/python vllm wrapt
import os, signal, subprocess, time, urllib.request

def start(cmd, log_path, env=None):
    return subprocess.Popen(cmd, stdout=open(log_path, "w"), stderr=subprocess.STDOUT,
                            env={**os.environ, **(env or {})}, start_new_session=True)

def wait_ready(url, log_path, minutes=30):
    for i in range(minutes * 4):
        try:
            urllib.request.urlopen(url, timeout=5); print(f"ready after ~{i*15}s"); return True
        except Exception:
            time.sleep(15)
    print("NOT ready -- log:"); os.system(f"tail -n 30 {log_path}"); return False

def stop(proc):
    os.killpg(proc.pid, signal.SIGTERM)
    try: proc.wait(timeout=120)
    except subprocess.TimeoutExpired: os.killpg(proc.pid, signal.SIGKILL)
    time.sleep(10)

def start_vllm(model, log_path, util=0.75):
    return start(["/kaggle/vllm-env/bin/vllm", "serve", model, "--tensor-parallel-size", "2",
                  "--dtype", "half", "--max-model-len", "8192", "--gpu-memory-utilization", str(util),
                  "--port", "8000"], log_path, env={"PYTHONPATH": ""})

In [ ]:
# Project dependencies (RAGAS pins openai<2 through instructor, so keep the project's pins)
!pip install -q -r requirements.txt
!pip install -q -e . --no-deps
!mkdir -p eval_out reports models

## 1 · Reranker distillation

In [ ]:
!python -m src.optimization.distill_reranker --device cuda:0 --questions data/evaluation/retrieval_eval.json build-data > eval_out/distill_data.log 2>&1; tail -n 3 eval_out/distill_data.log
!python -m src.optimization.distill_reranker --device cuda:0 train > eval_out/distill_train.log 2>&1; tail -n 6 eval_out/distill_train.log
!python -m src.optimization.distill_reranker --device cuda:0 --questions data/evaluation/retrieval_eval.json evaluate > eval_out/distill_eval.log 2>&1; cat reports/reranker_distillation.md

## 2 · AWQ generation (retrieval is cached so every label sees identical contexts)

In [ ]:
os.environ.update(MLFLOW_TRACKING_URI="sqlite:////kaggle/working/repo/mlflow.db", MLFLOW_DISABLE_AGENT_HINT="1")
awq = start_vllm("Qwen/Qwen3-8B-AWQ", "/kaggle/working/vllm_awq.log")
wait_ready("http://localhost:8000/v1/models", "/kaggle/working/vllm_awq.log")
!grep -iE "quantiz|Model loading took" /kaggle/working/vllm_awq.log | tail -n 5
# If vLLM complains about the quantization method, add "--quantization", "awq" to start_vllm and re-run.

In [ ]:
!python -m src.optimization.compare_llms generate --label awq --model Qwen/Qwen3-8B-AWQ --device cuda:1 \
    --vllm-log /kaggle/working/vllm_awq.log > eval_out/gen_awq.log 2>&1; tail -n 3 eval_out/gen_awq.log
!python -m src.optimization.compare_llms generate --label awq_student --model Qwen/Qwen3-8B-AWQ --device cuda:1 \
    --reranker models/reranker-student --vllm-log /kaggle/working/vllm_awq.log > eval_out/gen_awq_student.log 2>&1; tail -n 3 eval_out/gen_awq_student.log
stop(awq)

## 3 · fp16 baseline, then judge everything with the fp16 model

In [ ]:
fp16 = start_vllm("Qwen/Qwen3-8B", "/kaggle/working/vllm_fp16.log")
wait_ready("http://localhost:8000/v1/models", "/kaggle/working/vllm_fp16.log")
!python -m src.optimization.compare_llms generate --label fp16 --model Qwen/Qwen3-8B --device cuda:1 \
    --vllm-log /kaggle/working/vllm_fp16.log > eval_out/gen_fp16.log 2>&1; tail -n 3 eval_out/gen_fp16.log
!python -m src.optimization.compare_llms score --labels fp16 awq awq_student --judge-model Qwen/Qwen3-8B --device cuda:0 \
    > eval_out/score.log 2>&1; grep "\[score\]" eval_out/score.log

In [ ]:
!python -m src.optimization.compare_llms report --baseline fp16 --optimized awq --extra awq_student > eval_out/report.log 2>&1; cat reports/quantization.md

In [ ]:
stop(fp16)
!zip -qr /kaggle/working/optimization_results.zip reports mlflow.db mlruns eval_out/quant eval_out/*.log models/reranker-student
from IPython.display import FileLink
display(FileLink("/kaggle/working/optimization_results.zip"))